# 개별종목 조합J — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합J 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합J의 피처 값만 지정합니다.
import json

COMBINATION = 'J'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
    'relative_ret_5_market',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합J 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return', 'relative_ret_5_market')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.5043,0.5012,0.0031,0.3058,0.3607,0.0829,0.3819,0.0761,0.1632
1,2,balanced,980,20150123,20150421,0.3935,0.3978,-0.0043,0.3504,0.3638,0.0583,0.3783,0.1939,0.2843
2,3,balanced,1210,20151228,20160328,0.3614,0.3762,-0.0147,0.3598,0.3596,0.0425,0.3721,0.3400,0.3535
3,4,balanced,1439,20161202,20170228,0.4638,0.4617,0.0021,0.3655,0.3847,0.1022,0.4109,0.1687,0.2773
4,5,balanced,1669,20171113,20180207,0.4221,0.3901,0.0320,0.3887,0.3992,0.1092,0.3967,0.2818,0.3533
5,6,balanced,1899,20181024,20190118,0.4139,0.3725,0.0414,0.4132,0.4222,0.1361,0.4191,0.5259,0.4453
6,7,balanced,2129,20190930,20191224,0.4706,0.4781,-0.0075,0.3554,0.3776,0.0943,0.4081,0.1720,0.2790
7,8,balanced,2359,20200902,20201130,0.4105,0.3476,0.0629,0.4069,0.4104,0.1162,0.4062,0.4295,0.4154
8,9,balanced,2589,20210806,20211105,0.3841,0.3916,-0.0075,0.3737,0.3849,0.0760,0.3847,0.2832,0.3405
9,10,balanced,2818,20220714,20221012,0.3527,0.3454,0.0072,0.3526,0.3558,0.0357,0.3703,0.2887,0.3284


,OOS 폴드 평균
accuracy,0.4145
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0177
macro_f1,0.3712
balanced_accuracy,0.3845
mcc,0.0877
pr_auc_macro_ovr,0.3944
down_recall,0.2819
core_harmonic_mean,0.3304


재실행 명령: python scripts/run_stock_model_experiment.py
